> **対応するブログ記事**: [#7 Pythonで論文のFigure 1を再現する：相関行列・クラスタリング・PCA](../blog/article-07-visualization.md)
>
> このNotebookはブログ記事 #7 のコードをセルごとに実行できるインタラクティブ版です。各手法の使い分けやパラメータの詳しい解説はブログ記事を参照してください。

# Step 7: 全体像の可視化（Figure 1）

前処理済みデータから Figure 1 (a)~(c) を再現する。
- (a) 相関行列ヒートマップ
- (b) 教師なし階層的クラスタリング
- (c) PCA（主成分分析）

In [ ]:
# --- ライブラリのインポート ---
import numpy as np              # 数値計算ライブラリ（配列演算・線形代数等）
import pandas as pd             # データフレーム操作ライブラリ（CSV読込・集計等）
import matplotlib.pyplot as plt # グラフ描画ライブラリ（図の作成・保存）
import seaborn as sns           # 統計データ可視化ライブラリ（ヒートマップ・クラスタマップ等）
from sklearn.decomposition import PCA  # scikit-learnの主成分分析（PCA）クラス
from matplotlib.patches import Ellipse, Patch  # Ellipse: 楕円描画, Patch: 凡例用カラーパッチ
import matplotlib.transforms as transforms     # アフィン変換（楕円の回転・拡縮・平行移動）

# Jupyter Notebook内にグラフをインライン表示するマジックコマンド
%matplotlib inline

In [ ]:
# --- 出力先パスと配色の定数定義 ---
RESULTS = "../results"          # 前処理済みデータやCSVテーブルの保存先ディレクトリ
FIG = "../results/figures"      # 生成した図（PNG）の保存先ディレクトリ

COLOR_NORMAL = "#4EAED1"        # 非腫瘍（Normal）サンプルに割り当てる青色
COLOR_TUMOR = "#E8524A"         # 腫瘍（Tumor）サンプルに割り当てる赤色

In [ ]:
# --- データ読み込みとサンプル情報の準備 ---

# 前処理済みのタンパク質発現データを読み込む（行: タンパク質, 列: サンプル）
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)

# サンプル情報（Sample列, Condition列など）を読み込む
sample_info = pd.read_csv(f"{RESULTS}/sample_info.csv")

# Sample列をインデックスにし、各サンプルの条件（Normal/Tumor）をSeriesとして取得
conditions = sample_info.set_index("Sample")["Condition"]

# 条件に応じてサンプルごとの表示色をマッピング（Normal→青, Tumor→赤）
sample_colors = conditions.map({"Normal": COLOR_NORMAL, "Tumor": COLOR_TUMOR})

# データの形状を表示して読み込み結果を確認
print(f"データ: {df.shape[0]} タンパク質 x {df.shape[1]} サンプル")

## (a) 相関行列ヒートマップ

In [ ]:
def plot_correlation(df, sample_colors):
    """サンプル間のピアソン相関行列をクラスタリング付きヒートマップで描画する。"""

    # サンプル間のピアソン相関係数行列を計算（値域: -1〜1, 通常は0.7〜1.0の範囲）
    corr = df.corr(method="pearson")

    # 相関行列のインデックス順にサンプル色を並び替え（行カラーバー用）
    row_colors = sample_colors.reindex(corr.index)

    # クラスタリング付きヒートマップを描画
    g = sns.clustermap(
        corr,                           # 描画する相関行列データ
        method="average",               # UPGMA法（群平均法）で階層的クラスタリング
        metric="correlation",           # 距離指標: 1 - ピアソン相関を距離とする
        cmap="YlOrRd",                  # カラーマップ: 黄→オレンジ→赤（暖色系グラデーション）
        vmin=0.7, vmax=1.0,             # カラースケールの範囲を0.7〜1.0に制限
        figsize=(10, 10),               # 図のサイズ: 10×10インチ
        row_colors=row_colors,          # 行方向のカラーバー（Normal/Tumorの色分け）
        col_colors=row_colors,          # 列方向のカラーバー（行と同じ色分け）
        linewidths=0,                   # セル間の区切り線の太さ: 0（線なし）
        xticklabels=True,               # X軸のサンプル名ラベルを表示
        yticklabels=True,               # Y軸のサンプル名ラベルを表示
    )

    # X軸ラベルのフォントサイズと回転角度を設定（サンプル名が重ならないよう90度回転）
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=90)
    # Y軸ラベルのフォントサイズを設定
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)

    # 凡例用のカラーパッチを作成（Non-tumor: 青, Tumor: 赤）
    legend_elements = [Patch(facecolor=COLOR_NORMAL, label="Non-tumor"),
                       Patch(facecolor=COLOR_TUMOR, label="Tumor")]
    # ヒートマップの右上に凡例を配置（bbox_to_anchorで位置調整）
    g.ax_heatmap.legend(handles=legend_elements, loc="upper left",
                        bbox_to_anchor=(1.05, 1.0), frameon=False)

    # 図をPNG形式で保存（解像度150dpi, 余白を自動トリミング）
    g.savefig(f"{FIG}/fig1a_correlation.png", dpi=150, bbox_inches="tight")
    # メモリ解放のため図を閉じる
    plt.close()

    # 相関行列をCSVファイルとして保存（後続の分析や確認用）
    corr.to_csv(f"{RESULTS}/tables/correlation_matrix.csv")
    print("保存: fig1a_correlation.png")

# 関数を実行して相関行列ヒートマップを生成・保存
plot_correlation(df, sample_colors)

## (b) 階層的クラスタリング

In [ ]:
def plot_clustering(df, sample_info, sample_colors):
    """全タンパク質の発現量を用いた教師なし階層的クラスタリングヒートマップを描画する。"""

    # --- タンパク質ごとの発現方向（Up/Down in tumor）でカラーバーを作成 ---
    # Normalサンプル名の一覧を取得
    normals = sample_info[sample_info["Condition"] == "Normal"]["Sample"]
    # Tumorサンプル名の一覧を取得
    tumors = sample_info[sample_info["Condition"] == "Tumor"]["Sample"]

    # 各タンパク質について、Tumor平均 - Normal平均 = log2FC（発現変動の方向）を計算
    fc = (df[tumors.tolist()].mean(axis=1) - df[normals.tolist()].mean(axis=1))
    # fc > 0 なら腫瘍で上昇（赤）、fc <= 0 なら腫瘍で低下（青）で色分け
    protein_colors = fc.apply(lambda x: "#E74C3C" if x > 0 else "#3498DB")

    # クラスタリング付きヒートマップを描画（転置して サンプル(行) x タンパク質(列) にする）
    g = sns.clustermap(
        df.T,                    # 転置: サンプル(行) x タンパク質(列)
        method="ward",           # ウォード法: クラスタ内分散の増加を最小化する凝集法
        metric="euclidean",      # 距離指標: ユークリッド距離
        cmap="RdBu_r",           # カラーマップ: 赤=高発現, 青=低発現（反転版）
        center=0,                # カラーマップの中心値を0に設定
        z_score=1,               # 列(タンパク質)方向でZスコア標準化して相対パターンを可視化
        vmin=-3, vmax=3,         # カラースケールの範囲: Zスコア -3〜+3 に制限
        figsize=(14, 8),         # 図のサイズ: 横14×縦8インチ
        row_colors=sample_colors.reindex(df.columns),  # 行カラーバー: サンプルのNormal/Tumor色分け
        col_colors=protein_colors,  # 列カラーバー: タンパク質のUp/Down色分け
        xticklabels=False,       # X軸ラベル非表示（タンパク質数が多すぎるため）
        yticklabels=True,        # Y軸ラベル表示（サンプル名を表示）
    )

    # Y軸（サンプル名）のフォントサイズを設定
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
    # X軸のラベルを「Proteins」に設定
    g.ax_heatmap.set_xlabel("Proteins")
    # Y軸のラベルを「Samples」に設定
    g.ax_heatmap.set_ylabel("Samples")

    # 凡例用のカラーパッチを作成（腫瘍で上昇: 赤, 腫瘍で低下: 青）
    legend_elements = [Patch(facecolor="#E74C3C", label="Up-regulated in tumor"),
                       Patch(facecolor="#3498DB", label="Down-regulated in tumor")]
    # ヒートマップの右下に凡例を配置（bbox_to_anchorで位置微調整）
    g.ax_heatmap.legend(handles=legend_elements, loc="lower right",
                        bbox_to_anchor=(1.3, -0.15), frameon=False, fontsize=8)

    # 図をPNG形式で保存（解像度150dpi, 余白を自動トリミング）
    g.savefig(f"{FIG}/fig1b_clustering.png", dpi=150, bbox_inches="tight")
    # メモリ解放のため図を閉じる
    plt.close()
    print("保存: fig1b_clustering.png")

# 関数を実行してクラスタリングヒートマップを生成・保存
plot_clustering(df, sample_info, sample_colors)

## (c) PCA

In [ ]:
def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """2変量データの95%信頼楕円を描画する。
    
    共分散行列から楕円の形状を計算し、アフィン変換で
    正しい位置・スケール・回転を適用してAxesに追加する。
    """
    # データ点が2未満の場合は楕円を描画できないので終了
    if len(x) < 2:
        return

    # x, yの2×2共分散行列を計算（対角: 分散, 非対角: 共分散）
    cov = np.cov(x, y)

    # ピアソン相関係数 = 共分散 / (x標準偏差 * y標準偏差)
    # 楕円の傾きを決定する
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])

    # 楕円の半径を相関係数から計算（相関が高いほど細長い楕円になる）
    ell_radius_x = np.sqrt(1 + pearson)   # X方向の半径（1+r の平方根）
    ell_radius_y = np.sqrt(1 - pearson)   # Y方向の半径（1-r の平方根）

    # 原点中心の楕円オブジェクトを作成（幅=2*半径x, 高さ=2*半径y）
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2, **kwargs)

    # X方向のスケール = x標準偏差 × 標準偏差の数（n_std=2で約95%信頼区間）
    scale_x = np.sqrt(cov[0, 0]) * n_std
    # Y方向のスケール = y標準偏差 × 標準偏差の数
    scale_y = np.sqrt(cov[1, 1]) * n_std

    # アフィン変換を構成: 45度回転 → スケール適用 → データ中心に平行移動
    transf = (transforms.Affine2D()
              .rotate_deg(45)                          # 45度回転（共分散の方向に合わせる）
              .scale(scale_x, scale_y)                 # 標準偏差でスケーリング
              .translate(np.mean(x), np.mean(y)))      # データの平均値（重心）に移動

    # 楕円にアフィン変換+データ座標系の変換を設定
    ellipse.set_transform(transf + ax.transData)

    # Axesに楕円パッチを追加して返す
    return ax.add_patch(ellipse)

In [ ]:
def plot_pca(df, conditions):
    """主成分分析（PCA）を実行し、PC1 vs PC2 の散布図を描画する。"""

    # PCAオブジェクトを作成（第2主成分まで抽出）
    pca = PCA(n_components=2)
    # 転置してサンプル(行) x タンパク質(列)の形にしてPCAを実行、スコア行列を取得
    scores = pca.fit_transform(df.T)

    # 図と描画領域（Axes）を作成（サイズ: 横8×縦6インチ）
    fig, ax = plt.subplots(figsize=(8, 6))

    # 条件ごとの色マッピング辞書（Normal: 青, Tumor: 赤）
    cmap = {"Normal": "#3498DB", "Tumor": "#E74C3C"}

    # 条件ごとにループして散布図と信頼楕円を描画
    for cond, color in cmap.items():
        # 現在の条件に該当するサンプルのブールマスクを作成
        mask = conditions.reindex(df.columns) == cond

        # 散布図を描画（該当条件のサンプルのみ）
        ax.scatter(scores[mask, 0], scores[mask, 1],  # PC1(x), PC2(y)のスコア
                   c=color,            # 点の塗りつぶし色
                   s=100,              # 点のサイズ（ポイント^2単位）
                   alpha=0.8,          # 透明度（0=透明, 1=不透明）
                   label=cond,         # 凡例に表示するラベル
                   edgecolors="white", # 点の縁取り色（白）
                   linewidth=0.5)      # 縁取り線の太さ

        # 95%信頼楕円を描画（n_std=2.0 で約95%のデータ点を囲む）
        confidence_ellipse(scores[mask, 0], scores[mask, 1], ax, n_std=2.0,
                           facecolor=color,     # 楕円の塗りつぶし色
                           alpha=0.15,          # 楕円の透明度（薄く表示）
                           edgecolor=color,     # 楕円の輪郭色
                           linewidth=1.5)       # 楕円の輪郭線の太さ

    # 各主成分の寄与率（分散説明率）を取得
    ev = pca.explained_variance_ratio_

    # X軸ラベル: 第1主成分とその寄与率（%）
    ax.set_xlabel(f"Component 1 ({ev[0]*100:.1f}%)")
    # Y軸ラベル: 第2主成分とその寄与率（%）
    ax.set_ylabel(f"Component 2 ({ev[1]*100:.1f}%)")
    # 図のタイトルを設定
    ax.set_title("PCA of Non-tumor and Tumor Tissues")
    # 凡例を表示（枠線なし）
    ax.legend(frameon=False)
    # グリッド線を描画（灰色, 薄い透明度, 実線, 細い線幅）
    ax.grid(True, color="gray", alpha=0.3, linestyle="-", linewidth=0.5)
    # 上側の枠線（スパイン）を非表示にする
    ax.spines["top"].set_visible(False)
    # 右側の枠線（スパイン）を非表示にする
    ax.spines["right"].set_visible(False)

    # 図をPNG形式で保存（解像度150dpi, 余白を自動トリミング）
    fig.savefig(f"{FIG}/fig1c_pca.png", dpi=150, bbox_inches="tight")
    # メモリ解放のため図を閉じる
    plt.close()

    # PCAの分散説明率テーブルを作成（各主成分の寄与率と累積寄与率）
    var_df = pd.DataFrame({"PC": [f"PC{i+1}" for i in range(len(ev))],       # 主成分番号
                           "Variance_Ratio": ev,                              # 各PCの寄与率
                           "Cumulative": np.cumsum(ev)})                      # 累積寄与率
    # 分散説明率テーブルをCSVファイルとして保存
    var_df.to_csv(f"{RESULTS}/tables/pca_variance.csv", index=False)
    print("保存: fig1c_pca.png")

# 関数を実行してPCA散布図を生成・保存
plot_pca(df, conditions)

## 完了

Figure 1 (a)~(c) を `results/figures/` に保存した。